# Resemble Enhance (Denoise Only)

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
from denoise_utils.clear_memory import clear_memory
from denoise_utils.batch import batch_denoise, save_wav_torchaudio

warnings.filterwarnings('ignore')

In [ ]:
from denoise_utils.config import get_audio_files, get_output_dir

files_Pitt = get_audio_files('Pitt-origin')
out_Pitt = get_output_dir('Pitt-origin', 'Resemble')

files_Lu = get_audio_files('Lu')
out_Lu = get_output_dir('Lu', 'Resemble')

## Load Model

In [ ]:
from resemble_enhance.enhancer.inference import denoise as resemble_denoise

# Device detection
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print("Model will be auto-downloaded on first call (from HuggingFace)")

## Denoise Function

In [ ]:
def denoise_audio(audio_path, device):
    """
    Apply Resemble Enhance for speech denoising (denoise only)
    """
    dwav, sr = torchaudio.load(str(audio_path))

    # Multi-channel to mono
    dwav = dwav.mean(0)  # (channels, time) -> (time,)

    # resemble_denoise internally resamples to 44100Hz and processes
    hwav, out_sr = resemble_denoise(dwav=dwav, sr=sr, device=device, run_dir=None)

    return hwav, out_sr

## Pitt Denoise

In [ ]:
denoise_fn = lambda p: denoise_audio(p, device)

clear_memory()

batch_denoise(
    files_Pitt['Dementia'],
    out_Pitt / 'Dementia',
    denoise_fn,
    'Dementia',
    save_fn=save_wav_torchaudio,
)

clear_memory()

batch_denoise(
    files_Pitt['Control'],
    out_Pitt / 'Control',
    denoise_fn,
    'Control',
    save_fn=save_wav_torchaudio,
)

## Lu Denoise

In [ ]:
# denoise_fn = lambda p: denoise_audio(p, device)

# clear_memory()

# batch_denoise(
#     files_Lu['Dementia'],
#     out_Lu / 'Dementia',
#     denoise_fn,
#     'Dementia',
#     save_fn=save_wav_torchaudio,
# )

# clear_memory()

# batch_denoise(
#     files_Lu['Control'],
#     out_Lu / 'Control',
#     denoise_fn,
#     'Control',
#     save_fn=save_wav_torchaudio,
# )